### 322. 零钱兑换
给你一个整数数组 coins ，表示不同面额的硬币；以及一个整数 amount ，表示总金额。

计算并返回可以凑成总金额所需的 最少的硬币个数 。如果没有任何一种硬币组合能组成总金额，返回 -1 。

你可以认为每种硬币的数量是无限的。

示例 1：
- 输入：coins = [1, 2, 5], amount = 11
- 输出：3
- 解释：11 = 5 + 5 + 1

示例 2：
- 输入：coins = [2], amount = 3
- 输出：-1

示例 3：
- 输入：coins = [1], amount = 0
- 输出：0

提示：
- `1 <= coins.length <= 12`
- `1 <= coins[i] <= 2^31 - 1`
- `0 <= amount <= 10^4`


### 背包问题详解

#### 1.  01背包
有 n 种物品，第 i 种物品的体积为 w[i]，价值为 v[i]，背包容量为 c。每种物品只能选一个，求 体积 小于等于 c 的背包中物品的 最大价值。

01背包 回溯法“
1. 当前操作：枚举第 i 个物品选一个或不选。不选，剩余容量不变；选一个，剩余容量 减少w[i]。
2. 子问题：在剩余 容量为 c 时，从前 i 种物品中得到最大价值和。
3. 下一个子问题，分类讨论：
   1. 不选，在剩余容量为 c 时，从前 i-1 种物品中得到最大价值和。
   2. 选，在剩余容量为 c-w[i] 时（确保第 i个能被选，留下足够容积），从前 i-1 种物品中得到最大价值和。
4. 递归公式：`dfs[i][c] = max(dfs[i-1][c], dfs[i -1][c-w[i]] + v[i])`     # i 需要减一，01背包只能选一个

#### 2. 完全背包
有 n 种物品，第 i 种物品的体积为 w[i]，价值为 v[i]，背包容量为 c。每种物品无限次重复选，求 体积 小于等于 c 的背包中物品的 最大价值。

选与不选 回溯法“
1. 当前操作：枚举第 i 个物品选一个或不选。不选，剩余容量不变；选一个，剩余容量 减少w[i]。
2. 子问题：在剩余 容量为 c 时，从前 i 种物品中得到的最大价值和。
3. 下一个子问题，分类讨论：
   1. 不选，在剩余容量为 c 时，从前 i-1 种物品中得到最大价值和。
   2. 选一个，在剩余容量为 c-w[i] 时（确保第 i个能被选，留下足够容积），从前 i 种物品中得到最大价值和。
4. 递归公式：`dfs[i][c] = max(dfs[i-1][c], dfs[i][c-w[i]] + v[i])`     #  `dfs[i][c-w[i]] + v[i]`中 i 不变，完全背包可重复选
5. 初始条件：f[0][c] = 0
 
**常见变形**：
1. 至多装 擦capacity，求方案数/最大价值和
2. 至少装 capacity，求方案数/最小价值和
3. 恰好装 capacity，求方案数/最大/最小价值和： 最大变为最小，`dfs(i, c) = min(dfs(i-1, c), dfs(i, c-w[i]) + v[i])` 
 
- 时间复杂度：O(n*c)
- 空间复杂度：O(n*c)

In [ ]:
# 递归搜索 + 保存计算结果 = 记忆化搜索

from functools import cache
from typing import List

class Solution:
    def coinChange(self, coins: List[int], amount: int) -> int:
        # 视为完全 背包问题
        # coins 的数量视为价值，求最少数量即求最小价值
        # coins 的面额视为体积，amount 视为背包容积，即 c
        n = len(coins)

        @cache
        def dfs(i, c): 
            if i < 0: # 没有硬币了
                return 0 if c == 0 else float('inf')

            if c < coins[i]: # 容量不够了,只能不选
                return dfs(i - 1, c)
            # 不选 vs 选（数量 + 1）
            return min(dfs(i - 1, c), dfs(i, c - coins[i]) + 1) # i 不变表示可以重复选
        
        ans = dfs(n - 1, amount)
        return ans if ans < float('inf') else -1

coins = [1, 2, 5]
amount = 11
print(Solution().coinChange(coins, amount))

3


#### 2. 递推——动态规划
将上述递归公式 转换为动态规划的递推公式：
1. 使用 f[i][c] 二维数组存储状态。表示 前 i 个物品，放入容量为 c 的背包中，可获得的最大价值。
2. 记整数数组 coins 的长度为 n。为便于状态更新，减少对边界的判断，初始二维 dp 数组维度为 (n+1)×(∗)，其中第一维为 n+1 也意味着：第 i 种硬币为 coins[i−1]，第 1 种硬币为 coins[0]，第 0 种硬币为空。
3. 状态初始化：
   1. 初始化时，不合法的或未定义的状态则可以设置为正无穷或一个不可能取到的较大值： 
   2. dp[0][0]=0：表示从前 0 种硬币中选出若干个组成金额 0 所对应的最小硬币数目为 0，即「空集合」不选任何硬币即可得到金额 0。
   3. 对于 dp[0][j],  j≥1，则可将其设置为正无穷或一个不可能取到的较大值;
   4. 对于 dp[i][0],  i≥1，可将其设为 dp[i][0]=0，表示前 i 种硬币中，组成金额 0 所对应的最小硬币数目为 0。(这一点在程序的迭代实现中已有体现，可无需提前重复定义。)

4. 状态转移方程：`dp[i + 1][c] = min(dp[i][c], dp[i + 1][c- x]+1)`


In [ ]:

class Solution:
    def coinChange(self, coins: List[int], amount: int) -> int:
        n = len(coins)
        f = [[float('inf')] * (amount + 1) for _ in range(2)] # 只需要两行，滚动数组
        f[0][0] = 0 

        for i, x in enumerate(coins): # 枚举物品
            for c in range(amount + 1): # 枚举背包容量
                if c < x: # 容量不够了,只能不选
                    f[i + 1][c] = f[i][c]
                else:
                    f[i + 1][c] = min(f[i][c], f[i + 1][c - x] + 1)
        ans = f[n][amount]
        return ans if ans < float('inf') else -1